# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tessa-Saumu/FlyRank-ML-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

> **Freestyle submission.** This notebook belongs to my final, freestyle capstone: *which ranking pages are losing clicks to AI-answer search before it shows, and which should an editor protect first?* My earlier Lane 2 (refresh) experiments are kept for the record in `work/archive/` and are not part of this submission.

**What this card asks:** two signal checks with bucket tables, n and a verdict; ONE rule with a score, one reason code
and an action label, written to `work/outputs/baseline_action_score.csv`; and a review of the top 20. **The principle it
serves:** the solution hierarchy. A transparent rule comes first, and any model must beat it on the same rows.

In [1]:
# Setup: works locally and in Colab. Installs only missing packages and never prints a token.
import importlib.util, json, os, sys
if any(importlib.util.find_spec(p) is None for p in ["duckdb", "huggingface_hub", "sklearn", "scipy", "matplotlib"]):
    get_ipython().run_line_magic("pip", "install -q duckdb huggingface_hub pandas scikit-learn scipy matplotlib")
here = os.path.abspath(os.getcwd())
ROOT = next((p for p in [here, os.path.dirname(here), os.path.dirname(os.path.dirname(here))]
             if os.path.isdir(os.path.join(p, "work", "scripts"))), None)
if ROOT is None:  # fresh Colab runtime: fetch the code (no data lives in git)
    get_ipython().system("git clone -q https://github.com/Tessa-Saumu/FlyRank-ML-Internship.git")
    ROOT = os.path.abspath("FlyRank-ML-Internship")
sys.path.insert(0, os.path.join(ROOT, "work", "scripts"))
OUT, FIG = os.path.join(ROOT, "work", "outputs"), os.path.join(ROOT, "work", "figures")
import numpy as np, pandas as pd
import warnings; warnings.filterwarnings("ignore")
from IPython.display import Image, display, Markdown
import wh
DATA = wh.data_available(try_download=True)   # needs HF_TOKEN as an env var or Colab secret
print("pinned warehouse snapshot available:", DATA, "| revision", wh.REV[:8])
receipt = lambda name: json.load(open(os.path.join(OUT, name)))
pd.set_option("display.width", 170); pd.set_option("display.max_columns", 30)

pinned warehouse snapshot available: True | revision 50cbf7c3


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Question:** what rule would an editor write today, and which reason codes can it output? **Why it matters (FlyRank
principle: the solution hierarchy):** a transparent rule is the bar any model must clear.

**Rule:** `score = position tier (top 3 = 2, positions 4-6 = 1, 7-10 = 0) + max(0, -CTR change from month -2 to -1)`,
with a deterministic hash tiebreak. It encodes two common intuitions: AI answers sit at the top, so top positions are
most exposed, and a CTR that has started slipping keeps slipping.

**Reason codes it can output:** `TOP3_SLIPPING` (top-3 position and CTR down more than 5% last month), `TOP3_EXPOSED`
(top-3 position, CTR not falling) and `TOP10_SLIPPING` (positions 4 to 10). The action label is `review now`.

I froze the rule **before** looking at the checks, on purpose. It represents the status-quo heuristic, so it must not be
fitted to the data. The checks then test whether its two intuitions hold.

**Two signal checks (training-era months, December and January).**

In [2]:
if DATA:
    from erosion_model import load, make_features
    df, grid, vals = load(); df = df[df.measurable.fillna(False)].reset_index(drop=True)
    X = make_features(df, grid, vals)
    d = pd.DataFrame({"origin": df.origin, "y": df.erode, "pos": X.pos_f, "g": X.ctr_growth})
    d = d[d.origin != "2026-05"]
    d["era"] = np.where(d.origin.isin(["2025-12", "2026-01"]), "design_dec_jan", "later_feb_apr")
    d["tier"] = pd.cut(d.pos, [0, 3.5, 6.5, 10.5], labels=["1-3", "4-6", "7-10"])
    d["ctr_change"] = pd.cut(d.g, [-9, -0.3, -0.05, 0.05, 0.3, 9], labels=["fell >26%", "fell 5-26%", "flat", "rose 5-35%", "rose >35%"])
    out = {}
    for key, col in [("check1_position_tier", "tier"), ("check2_recent_ctr_change", "ctr_change")]:
        t = d.groupby(["era", col], observed=True).y.agg(n="size", erosion_rate="mean").round(3).reset_index()
        out[key] = json.loads(t.to_json(orient="records"))
    json.dump(out, open(os.path.join(OUT, "signal_checks.json"), "w"), indent=1)
sc = receipt("signal_checks.json")
for k in sc:
    print(k); display(pd.DataFrame(sc[k]).pivot(index=sc[k] and list(sc[k][0])[1], columns="era", values=["n", "erosion_rate"]))

check1_position_tier


n                 erosion_rate              
era  design_dec_jan later_feb_apr design_dec_jan later_feb_apr
tier                                                          
1-3          2917.0        8577.0          0.266         0.282
4-6          5929.0       12808.0          0.279         0.305
7-10         2317.0        5497.0          0.310         0.299

check2_recent_ctr_change


n                 erosion_rate              
era        design_dec_jan later_feb_apr design_dec_jan later_feb_apr
ctr_change                                                          
fell 5-26%         2711.0        5498.0          0.245         0.241
fell >26%          3958.0        8095.0          0.254         0.307
flat               1109.0        3409.0          0.243         0.204
rose 5-35%         1933.0        5097.0          0.325         0.289
rose >35%          1452.0        4783.0          0.399         0.417

**Check 1: are top positions more exposed?** Tied to the position tiers FlyRank's own report uses (CTR by tier: top 3,
positions 4-10). In the design months, top-3 pages eroded at 0.266, positions 4-6 at 0.279 and positions 7-10 at 0.310
(n = 2,917 / 5,929 / 2,317). **Verdict: OPPOSITE.** Over a two-month horizon, top pages are *less* likely to lose
further clicks. The large fall in top-3 CTR in ML-06 is a level shift that has already happened across the panel, not a
page-level warning sign.

**Check 2: does a slipping CTR keep slipping?** Pages whose CTR rose by more than 35% eroded at 0.399. Pages whose CTR
fell by more than 26% eroded at 0.254 (n = 1,452 and 3,958). **Verdict: OPPOSITE.** A recent bump reverts. In the later
months the falling group rises to 0.307 while bumps stay highest (0.417), so the pattern is not fully stable.

**What it means.** Both intuitions in the frozen rule fail the training-era data. I keep the rule as the baseline
anyway, because it is what an editor would plausibly do without a model. That is exactly the comparison the model has
to win. ML-08 also compares a data-informed one-signal rule (CTR above peers), a client-shock rule and a persistence
rule, so the model is never only beating a weak strawman.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

**Question:** what does the rule produce when it ranks every eligible page? **Why it matters:** a ranked queue with one
reason code per row is what an editor can act on. **Method:** score every July 2026 page, rank, assign the reason code
and write `work/outputs/baseline_action_score.csv` (row level, gitignored).

In [3]:
if DATA:
    import erosion_playbook; erosion_playbook.main()
bq = pd.read_csv(os.path.join(OUT, "baseline_action_score.csv")) if os.path.exists(os.path.join(OUT, "baseline_action_score.csv")) else None
bm = receipt("baseline_metrics.json")
print("rule:", bm["rule"]); print("reason codes in the July 2026 rule queue:", bm["live_queue_reason_codes"])
if bq is not None:
    print("rows written to work/outputs/baseline_action_score.csv (gitignored):", len(bq))

{
 "decision_origin": "2026-07",
 "feature_window": [
  "2026-05",
  "2026-06"
 ],
 "trained_on_origins": [
  "2025-12",
  "2026-01",
  "2026-02",
  "2026-03",
  "2026-04",
  "2026-05"
 ],
 "train_rows": 49022,
 "lr_C": 1.0,
 "eligible_pages": 14100,
 "excluded_younger_than_90_days": 3243,
 "queue_size": 50,
 "queue_clients": 7,
 "max_pages_one_client": 10,
 "reason_codes": {
  "CTR_ABOVE_PEERS": 40,
  "DEMAND_SHIFT": 10
 },
 "queue_median_position": 4.8,
 "clicks_at_risk_2_months_if_25pct_lost": 25324,
 "score_quantiles_eligible": {
  "0.5": 0.2291,
  "0.9": 0.4285,
  "0.99": 0.8354
 },
 "queue_min_score": 0.8037,
 "client_cap": 10,
 "cap_cost_forward_origins": {
  "2026-02": {
   "uncapped_p@50": 0.52,
   "capped_p@50": 0.54,
   "uncapped_clients": 6,
   "capped_clients": 8
  },
  "2026-03": {
   "uncapped_p@50": 0.6,
   "capped_p@50": 0.62,
   "uncapped_clients": 11,
   "capped_clients": 11
  },
  "2026-04": {
   "uncapped_p@50": 0.66,
   "capped_p@50": 0.66,
   "uncapped_clients": 

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

**Question:** do the rule's top 20 make sense to a person? **Why it matters (FlyRank principle: a ranking is only useful
if its top rows are explainable):** for each of the rule's top 20 pages at the July 2026 decision point, I record the
action, the reason code, why the rule picked it, and what would make the pick wrong. Pages are shown by rank, never by ID.

In [4]:
if DATA:
    from erosion_model import load, make_features, rule_score, tiebreak
    df, grid, vals = load(); X = make_features(df, grid, vals); live = df.origin == "2026-07"
    t = pd.DataFrame({"position": X.pos_f[live].round(1), "ctr_change_pct": ((np.exp(X.ctr_growth[live]) - 1) * 100).round(0),
                      "ctr_vs_peers_pct": ((np.exp(X.ctr_vs_client[live]) - 1) * 100).round(0),
                      "clicks_last_month": df.clk_m1[live].fillna(0).astype(int),
                      "score": rule_score(X[live]) + tiebreak(df.content_hash_id[live])}).sort_values("score", ascending=False).head(20)
    t.insert(0, "rank", range(1, 21))
    t["reason_code"] = np.where(t.position <= 3.5, np.where(t.ctr_change_pct < -5, "TOP3_SLIPPING", "TOP3_EXPOSED"), "TOP10_SLIPPING")
    t["action"] = "review now"
    t["why"] = [f"position {p}, CTR {c:+.0f}% last month" for p, c in zip(t.position, t.ctr_change_pct)]
    t["what_would_make_it_wrong"] = np.where(t.ctr_change_pct < -50, "a tracking gap or demand drop, not AI answers (check impressions and season)",
                                    np.where(t.ctr_vs_peers_pct < 0, "the page already earns less than its position; the fall may be over",
                                             "a recent bump that is now stable; ML-07 check 2 says bumps tend to revert"))
    t.drop(columns="score").to_json(os.path.join(OUT, "baseline_top20_review.json"), orient="records", indent=1)
top20 = pd.read_json(os.path.join(OUT, "baseline_top20_review.json"))
display(top20)

,rank,position,ctr_change_pct,ctr_vs_peers_pct,clicks_last_month,reason_code,action,why,what_would_make_it_wrong
0,1,2.4,-99,118,366,TOP3_SLIPPING,review now,"position 2.4, CTR -99% last month","a tracking gap or demand drop, not AI answers ..."
1,2,1.8,-99,94,46,TOP3_SLIPPING,review now,"position 1.8, CTR -99% last month","a tracking gap or demand drop, not AI answers ..."
2,3,2.2,-99,60,57,TOP3_SLIPPING,review now,"position 2.2, CTR -99% last month","a tracking gap or demand drop, not AI answers ..."
3,4,2.7,-99,63,36,TOP3_SLIPPING,review now,"position 2.7, CTR -99% last month","a tracking gap or demand drop, not AI answers ..."
4,5,0.5,-97,222,76,TOP3_SLIPPING,review now,"position 0.5, CTR -97% last month","a tracking gap or demand drop, not AI answers ..."
5,6,3.2,-99,32,75,TOP3_SLIPPING,review now,"position 3.2, CTR -99% last month","a tracking gap or demand drop, not AI answers ..."
6,7,2.8,-98,189,87,TOP3_SLIPPING,review now,"position 2.8, CTR -98% last month","a tracking gap or demand drop, not AI answers ..."
7,8,1.7,-98,87,35,TOP3_SLIPPING,review now,"position 1.7, CTR -98% last month","a tracking gap or demand drop, not AI answers ..."
8,9,2.6,-99,93,97,TOP3_SLIPPING,review now,"position 2.6, CTR -99% last month","a tracking gap or demand drop, not AI answers ..."
9,10,3.4,-99,72,21,TOP3_SLIPPING,review now,"position 3.4, CTR -99% last month","a tracking gap or demand drop, not AI answers ..."


**What the review shows.** The rule's top 20 are all top-3 pages (positions 0.5 to 3.4) whose CTR fell 94% to 99% in one
month, and every one earns more than its client's peers at that position. A one-month collapse of that size is the kind
of change that tracking gaps or demand drops produce, so the last column sends the editor to impressions and season
before anything else. That is the failure pattern the model needs to avoid.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Question:** which picks look wrong, and did any product flag or future window leak in? **Why it matters (FlyRank
principle: attack your own baseline):** a baseline that is quietly weak or leaky makes any model look better than it is.

**Weak picks.**
- **Likely tracking or demand events.** All 20 reviewed picks carry the note "a tracking gap or demand drop, not AI
  answers", because CTR fell by more than 50% in one month.
- **Repeated rows.** The last three of the 20 share identical values (position, CTR change, and last-month clicks), so
  I would review them together, not as three independent finds.
- **A tie problem.** In three of four forward months all 50 picks share one score, because the CTR-fall term is capped
  and many top-3 pages hit the cap (their CTR collapsed). The hash tiebreak then picks 50 of them, effectively at
  random, so the rule's precision is the erosion rate of that tied group. I keep the rule frozen as written, because
  changing it after seeing results would defeat its purpose, and I report this as the same weakness my Lane 2 rule had.
  The distinct-score column below shows it.

In [5]:
display(pd.DataFrame(bm["per_origin"]).T)

,base_rate,p@20,p@50,p@100,ap,auc,distinct_scores_in_top50
2026-02,0.2273,0.40,0.46,0.41,0.1985,0.3925,3.0
2026-03,0.3382,0.60,0.76,0.70,0.3851,0.5005,1.0
2026-04,0.3079,0.50,0.50,0.57,0.3707,0.5519,1.0
2026-05,0.2397,0.35,0.38,0.35,0.2342,0.4526,1.0


**What it means.** The rule ranks above the base rate at 50 pages (0.38 to 0.76), but its whole-ranking quality is
near or below the base rate in two of four months, and on the sealed month its AUC is 0.45, worse than random order.
This is the baseline the model has to beat.

**Leakage check.** The rule reads only two features, the impression-weighted position and the log CTR change between
the two closed feature months. Neither contains an outcome-window value, a product flag (the release has none) or a
future month.

In [6]:
feats = receipt("erosion_features.json")
print({f: feats[f] for f in ["pos_f", "ctr_growth"]})
print("outcome-window or flag columns among the rule's inputs: none (the rule uses only the two features above)")

{'pos_f': 'impression-weighted position, feature window', 'ctr_growth': 'log CTR month -1 vs month -2 (clicks growth minus impressions growth)'}
outcome-window or flag columns among the rule's inputs: none (the rule uses only the two features above)


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.